In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install("ultralytics")
install("scikit-learn")
print("✅ Dependencies installed.")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import time, csv, os, sys, warnings
warnings.filterwarnings("ignore")

from datetime import datetime
from collections import deque
from dataclasses import dataclass, field
from typing import Optional

sys.argv = ["kaggle_kernel"]

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
    print("✅ YOLO loaded.")
except ImportError:
    YOLO_AVAILABLE = False
    print("⚠️  YOLO unavailable — simulation mode.")

KAGGLE_WORKING   = "/kaggle/working"
OUTPUT_CSV       = os.path.join(KAGGLE_WORKING, "near_miss_results.csv")
OUTPUT_SUMMARY   = os.path.join(KAGGLE_WORKING, "near_miss_summary.csv")
OUTPUT_VIDEO     = os.path.join(KAGGLE_WORKING, "near_miss_output.mp4")
OUTPUT_VIDEO_AVI = os.path.join(KAGGLE_WORKING, "near_miss_output.avi")
OUTPUT_ACCURACY  = os.path.join(KAGGLE_WORKING, "model_accuracy_report.csv")

DATASET_BASE        = "/kaggle/input/datasets/tharunravilla/near-miss-incident1"
NORMAL_DATASET_BASE = "/kaggle/input/datasets/tharunravilla/normal-videos-micro"

def _collect_videos(folder):
    exts = {".mp4", ".avi", ".mov", ".mkv",
            ".MP4", ".AVI", ".MOV", ".MKV"}
    found = []
    if not os.path.isdir(folder):
        print(f"  ❌ Folder not found: {folder}")
        return found
    for root, dirs, files in os.walk(folder):
        for f in sorted(files):
            if os.path.splitext(f)[1] in exts:
                found.append(os.path.join(root, f))
    return found

NEAR_MISS_VIDEO_PATHS = [
    os.path.join(DATASET_BASE, "near_miss/near_miss/outputvideos.new.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/VID-20260416-WA0001.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/AZ2Z3gZRcdzqW11H65twqQ-AZ2Z3gZRunELahKnJthr2w.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/VID-20260417-WA0010.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/AZ2Z4inmToeOnCKLG8Akwg-AZ2Z4inmDQ40ZzXmJ5F9JQ.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/Chain_Snatching13.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/AZ2Z7kmYFgfBv7xmX3Fpww-AZ2Z7kmYcHQ9mO5UkLi0WQ_110931.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/AZ2Z56_Rnt_MMXteGOzFNg-AZ2Z56_RPSaBD-QfRwdR2w.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/a8fcab479656d6a6c04b4f00d1ff5112.mp4"),
    os.path.join(DATASET_BASE, "video dataset of near miss/CCTV_Chain_Snatching_Attempt.mp4"),
]

NORMAL_VIDEO_PATHS = _collect_videos(NORMAL_DATASET_BASE)
ALL_VIDEO_PATHS    = NEAR_MISS_VIDEO_PATHS + NORMAL_VIDEO_PATHS

GROUND_TRUTH_LABELS = {
    "outputvideos.new.mp4":                                                  1,
    "VID-20260416-WA0001.mp4":                                               1,
    "AZ2Z3gZRcdzqW11H65twqQ-AZ2Z3gZRunELahKnJthr2w.mp4":                   1,
    "VID-20260417-WA0010.mp4":                                               1,
    "AZ2Z4inmToeOnCKLG8Akwg-AZ2Z4inmDQ40ZzXmJ5F9JQ.mp4":                   1,
    "Chain_Snatching13.mp4":                                                 1,
    "AZ2Z7kmYFgfBv7xmX3Fpww-AZ2Z7kmYcHQ9mO5UkLi0WQ_110931.mp4":           1,
    "AZ2Z56_Rnt_MMXteGOzFNg-AZ2Z56_RPSaBD-QfRwdR2w.mp4":                   1,
    "a8fcab479656d6a6c04b4f00d1ff5112.mp4":                                  1,
    "CCTV_Chain_Snatching_Attempt.mp4":                                      1,
}
for _vp in NORMAL_VIDEO_PATHS:
    GROUND_TRUTH_LABELS[os.path.basename(_vp)] = 0

CONFIG = {
    # ── Distance ──────────────────────────────────────────────────────
    "dist_threshold_m":           1.8,    # must be physically very close
    "pixels_per_meter":           60,

    # ── Motion ────────────────────────────────────────────────────────
    "motion_speed_threshold":     0.08,
    "neck_zone_ratio":            0.25,

    # ── Timing ────────────────────────────────────────────────────────
    "interaction_min_sec":        0.5,
    "interaction_max_sec":        6.0,

    # ── Victim qualification ──────────────────────────────────────────
    "reaction_move_px":           10,
    "reaction_frames":            8,
    "rider_overlap_thresh":       0.25,
    "require_vehicle_for_alert":  True,
    "victim_max_velocity_px":     4.0,    # victim must be nearly stationary
    "person_max_aspect_ratio":    0.75,
    "victim_min_track_frames":    12,     # must be tracked for 12+ frames

    # ── Suspect qualification ─────────────────────────────────────────
    "suspect_min_track_frames":   8,      # NEW: vehicle must be tracked 8+ frames
    "suspect_min_speed_px":       2.0,    # NEW: vehicle must actually be moving
    "suspect_speed_ratio_min":    1.5,
    "parallel_road_ratio":        1.8,    # horizontal dominance = traffic
    "approach_dot_min":           30.0,   # minimum dot product for approach
    "min_approach_frames":        4,      # must approach for 4 consecutive frames

    # ── Scoring ───────────────────────────────────────────────────────
    "min_conditions":             5,
    "min_confidence":             0.50,
    "min_alert_persist_frames":   4,
    "alert_cooldown_sec":         5.0,
    "raw_score_threshold":        0.62,

    # ── System ────────────────────────────────────────────────────────
    "fps":                        30,
    "display_width":              1280,
    "display_height":             720,
    "yolo_model":                 "yolo11n.pt",
    "yolo_conf":                  0.35,
}

PERSON_CLASS     = 0
BICYCLE_CLASS    = 1
CAR_CLASS        = 2
MOTORBIKE_CLASS  = 3
RELEVANT_CLASSES = {PERSON_CLASS, BICYCLE_CLASS, MOTORBIKE_CLASS, CAR_CLASS}
VEHICLE_CLASSES  = {BICYCLE_CLASS, MOTORBIKE_CLASS, CAR_CLASS}

print("✅ Configuration loaded.")
print(f"📂 Near-miss videos  : {len(NEAR_MISS_VIDEO_PATHS)}")
print(f"📂 Normal videos     : {len(NORMAL_VIDEO_PATHS)}")
print(f"📂 Total configured  : {len(ALL_VIDEO_PATHS)}")

In [ ]:
@dataclass
class Detection:
    track_id: int
    cls: int
    x1: float; y1: float; x2: float; y2: float
    conf: float

    @property
    def cx(self):     return (self.x1 + self.x2) / 2
    @property
    def cy(self):     return (self.y1 + self.y2) / 2
    @property
    def w(self):      return self.x2 - self.x1
    @property
    def h(self):      return self.y2 - self.y1
    @property
    def center(self): return (self.cx, self.cy)
    @property
    def aspect(self): return self.w / max(1.0, self.h)   # FIX-D


@dataclass
class TrackHistory:
    track_id: int
    positions:   deque = field(default_factory=lambda: deque(maxlen=90))
    last_seen:   int   = 0
    alert_count: int   = 0
    frame_count: int   = 0   # FIX-E: track age

    def update(self, cx, cy, frame_idx):
        self.positions.append((cx, cy, frame_idx))
        self.last_seen   = frame_idx
        self.frame_count += 1

    def velocity(self):
        if len(self.positions) < 2:
            return 0.0
        (x1,y1,_),(x2,y2,_) = self.positions[-2], self.positions[-1]
        return np.hypot(x2-x1, y2-y1)

    def avg_velocity(self, n=5):
        """Average velocity over last n frames — more stable than single-frame."""
        pts = list(self.positions)
        if len(pts) < 2:
            return 0.0
        recent = pts[max(0, len(pts)-n):]
        speeds = []
        for i in range(1, len(recent)):
            x1,y1,_ = recent[i-1]
            x2,y2,_ = recent[i]
            speeds.append(np.hypot(x2-x1, y2-y1))
        return float(np.mean(speeds)) if speeds else 0.0

    def displacement(self, n=5):
        if len(self.positions) < 2:
            return 0.0
        idx = max(0, len(self.positions)-n)
        x1,y1,_ = self.positions[idx]
        x2,y2,_ = self.positions[-1]
        return np.hypot(x2-x1, y2-y1)


@dataclass
class IncidentEvent:
    timestamp: str; frame_idx: int; zone: str
    dist_m: float; motion_speed: float; duration_sec: float
    reaction_conf: float; hand_to_neck: bool
    conditions_met: int; overall_conf: float
    status: str; alert: int; victim_id: int

print("✅ Data structures defined.")

In [ ]:
def draw_near_miss_banner(frame, conf: float):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 76), (0, 0, 150), -1)
    cv2.addWeighted(overlay, 0.80, frame, 0.20, 0, frame)
    cv2.putText(frame, "  NEAR-MISS / SNATCHING DETECTED",
                (16, 52), cv2.FONT_HERSHEY_DUPLEX, 1.1,
                (255, 255, 255), 2, cv2.LINE_AA)
    txt = f"Conf: {conf*100:.0f}%"
    (tw, _), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.82, 2)
    cv2.putText(frame, txt, (w - tw - 18, 52),
                cv2.FONT_HERSHEY_SIMPLEX, 0.82, (0, 220, 255), 2, cv2.LINE_AA)


def draw_suspicious_banner(frame, conf: float):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 64), (0, 85, 185), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
    cv2.putText(frame, "  SUSPICIOUS ACTIVITY",
                (16, 44), cv2.FONT_HERSHEY_DUPLEX, 1.0,
                (255, 255, 255), 2, cv2.LINE_AA)
    txt = f"Conf: {conf*100:.0f}%"
    (tw, _), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.75, 2)
    cv2.putText(frame, txt, (w - tw - 18, 44),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 220, 255), 2, cv2.LINE_AA)


def draw_victim_box(frame, det: Detection):
    x1,y1,x2,y2 = int(det.x1),int(det.y1),int(det.x2),int(det.y2)
    cv2.rectangle(frame, (x1-3, y1-3), (x2+3, y2+3), (0, 0, 110), 5)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
    bar_y = max(80, y1-30)
    cv2.rectangle(frame, (x1, bar_y), (x2, y1), (0, 0, 195), -1)
    cv2.putText(frame, f"VICTIM #{det.track_id}",
                (x1+5, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.70,
                (255, 255, 255), 2, cv2.LINE_AA)


def draw_suspect_box(frame, det: Detection):
    x1,y1,x2,y2 = int(det.x1),int(det.y1),int(det.x2),int(det.y2)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 140, 255), 3)
    bar_y = max(80, y1-30)
    cv2.rectangle(frame, (x1, bar_y), (x2, y1), (0, 90, 195), -1)
    type_name = {
        BICYCLE_CLASS:  "Bicycle",
        MOTORBIKE_CLASS:"Bike",
        CAR_CLASS:      "Vehicle",
        PERSON_CLASS:   "Person",
    }.get(det.cls, "Suspect")
    cv2.putText(frame, f"SUSPECT {type_name} #{det.track_id}",
                (x1+5, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                (255, 255, 255), 2, cv2.LINE_AA)


def draw_normal_person_box(frame, det: Detection):
    x1,y1,x2,y2 = int(det.x1),int(det.y1),int(det.x2),int(det.y2)
    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,200,0), 2)
    cv2.putText(frame, f"Person #{det.track_id}",
                (x1, y1-7), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (0,200,0), 1, cv2.LINE_AA)


def draw_distance_line(frame, victim: Detection, suspect: Detection,
                        dist_m: float, is_close: bool):
    vx, vy = int(victim.cx),  int(victim.cy)
    sx, sy = int(suspect.cx), int(suspect.cy)
    color  = (0, 0, 255) if is_close else (130, 130, 130)
    thick  = 2 if is_close else 1
    cv2.line(frame, (vx, vy), (sx, sy), color, thick)
    mid_x = (vx + sx) // 2
    mid_y = (vy + sy) // 2
    txt   = f"{dist_m:.2f} m"
    (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.68, 2)
    cv2.rectangle(frame,
                  (mid_x-5, mid_y-th-6), (mid_x+tw+5, mid_y+5),
                  (0, 0, 0), -1)
    cv2.putText(frame, txt, (mid_x, mid_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.68, color, 2, cv2.LINE_AA)


_artifact_cache = {}

def remove_baked_artifacts(frame):
    key = frame.shape
    if key in _artifact_cache:
        return frame
    h, w = frame.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    mask[555:718, 8:240] = 255
    hsv        = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    green_mask = cv2.inRange(hsv,
                             np.array([40,  80,  80], np.uint8),
                             np.array([80, 255, 255], np.uint8))
    k          = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    green_mask = cv2.dilate(green_mask, k, iterations=1)
    mask       = cv2.bitwise_or(mask, green_mask)
    mask[8:38, 295:450] = 255
    result     = cv2.inpaint(frame, mask, 7, cv2.INPAINT_TELEA)
    _artifact_cache[key] = True
    return result


print("✅ Annotation functions defined.")

In [ ]:
class SimulatedDetector:
    def __init__(self):
        self._frame   = 0
        self._ped_cx  = 300.0
        self._bike_cx = 620.0
        self._closing = True

    def detect(self, _frame_img):
        self._frame += 1
        if self._closing:        self._bike_cx -= 5
        if self._bike_cx < 240:  self._closing = False
        if not self._closing:    self._bike_cx += 7
        if self._bike_cx > 660:  self._closing = True
        return [
            Detection(1, PERSON_CLASS,
                      self._ped_cx-30, 150, self._ped_cx+30, 380, 0.92),
            Detection(2, MOTORBIKE_CLASS,
                      self._bike_cx-55, 170, self._bike_cx+55, 340, 0.88),
        ]

print("✅ SimulatedDetector ready.")

In [ ]:
class NearMissDetector:

    def __init__(self, source, save_csv=OUTPUT_CSV,
                 zone="Zone-A", save_video=True, video_index=0):
        self.source      = source
        self.save_csv    = save_csv
        self.zone        = zone
        self.save_video  = save_video
        self.video_index = video_index
        self.cfg         = CONFIG

        self.track_histories: dict  = {}
        self.events: list           = []
        self.frame_idx              = 0
        self.last_alert_frame       = -9999
        self.prev_gray              = None
        self.interaction_start      = None

        self.total_alerts     = 0
        self.total_suspicious = 0
        self.total_clear      = 0

        self.frame_confidences: list = []
        self.frame_predictions: list = []

        self._consecutive_near_miss  = 0
        self._flush_counter          = 0
        self._video_writer           = None
        self._actual_out_path        = None
        self._approach_counters: dict = {}   # (vid, pid) → consecutive approach frames

        if YOLO_AVAILABLE:
            print(f"[INFO] Loading YOLO: {self.cfg['yolo_model']}")
            self.model = YOLO(self.cfg["yolo_model"])
        else:
            print("[INFO] Using SimulatedDetector.")
            self.model = SimulatedDetector()

        self._csv_file   = None
        self._csv_writer = None
        if self.save_csv:
            self._open_csv(self.save_csv)

    # ── CSV helpers ────────────────────────────────────────────────────

    def _open_csv(self, path):
        write_header = not os.path.exists(path)
        self._csv_file   = open(path, "a", newline="")
        self._csv_writer = csv.writer(self._csv_file)
        if write_header:
            self._csv_writer.writerow([
                "Timestamp","Frame","Zone","VideoFile","Distance_m","MotionSpeed",
                "Duration_s","ReactionConf_%","HandToNeck","ConditionsMet",
                "OverallConf_%","Status","Alert","VictimID",
            ])

    def _write_csv_row(self, ev: IncidentEvent, video_file: str = ""):
        if self._csv_writer:
            self._csv_writer.writerow([
                ev.timestamp, ev.frame_idx, ev.zone, video_file,
                round(ev.dist_m, 3), round(ev.motion_speed, 3),
                round(ev.duration_sec, 3), round(ev.reaction_conf*100, 1),
                "YES" if ev.hand_to_neck else "NO",
                ev.conditions_met, round(ev.overall_conf*100, 1),
                ev.status, ev.alert, ev.victim_id,
            ])
            self._flush_counter += 1
            if self._flush_counter >= 30:
                self._csv_file.flush()
                self._flush_counter = 0

    # ── STEP 1: Object Detection ───────────────────────────────────────

    def _step1_detect(self, frame) -> list:
        if YOLO_AVAILABLE:
            results = self.model.track(
                frame, persist=True,
                conf=self.cfg["yolo_conf"],
                classes=list(RELEVANT_CLASSES),
                imgsz=640,
                verbose=False,
            )
            detections = []
            if results and results[0].boxes is not None:
                boxes = results[0].boxes
                for i in range(len(boxes)):
                    cls  = int(boxes.cls[i])
                    conf = float(boxes.conf[i])
                    tid  = int(boxes.id[i]) if boxes.id is not None else i
                    x1,y1,x2,y2 = boxes.xyxy[i].tolist()
                    detections.append(Detection(tid, cls, x1, y1, x2, y2, conf))
            return detections
        else:
            return self.model.detect(frame)

    # ── STEP 2: Identify Victim & Suspect ─────────────────────────────

    @staticmethod
    def _box_overlap_ratio(p: Detection, v: Detection) -> float:
        ix1 = max(p.x1, v.x1); iy1 = max(p.y1, v.y1)
        ix2 = min(p.x2, v.x2); iy2 = min(p.y2, v.y2)
        if ix2 <= ix1 or iy2 <= iy1:
            return 0.0
        inter  = (ix2 - ix1) * (iy2 - iy1)
        p_area = max(1.0, p.w * p.h)
        return inter / p_area

    def _is_valid_pedestrian(self, p: Detection) -> bool:
        """Victim must be slow, stable, upright tracked pedestrian."""
        if p.aspect > self.cfg["person_max_aspect_ratio"]:
            return False
        hist = self.track_histories.get(p.track_id)
        if hist is None:
            return False
        if hist.frame_count < self.cfg["victim_min_track_frames"]:
            return False
        if hist.avg_velocity(n=5) > self.cfg["victim_max_velocity_px"]:
            return False
        return True

    def _is_valid_suspect_vehicle(self, v: Detection) -> bool:
        """
        KEY FIX: Vehicle must be tracked long enough AND actually moving
        to qualify as a suspect. Eliminates briefly-seen passing cars.
        """
        hist = self.track_histories.get(v.track_id)
        if hist is None:
            return False
        # Must have track history — not a new detection
        if hist.frame_count < self.cfg.get("suspect_min_track_frames", 8):
            return False
        # Must actually be moving — parked/stationary cars are not suspects
        spd = hist.avg_velocity(n=5)
        if spd < self.cfg.get("suspect_min_speed_px", 2.0):
            return False
        return True

    def _is_vehicle_approaching_pedestrian(self, v: Detection, p: Detection) -> bool:
        """Vehicle trajectory must point meaningfully toward pedestrian."""
        v_hist = self.track_histories.get(v.track_id)
        if v_hist is None or len(v_hist.positions) < 6:
            return False
        pts = list(v_hist.positions)
        old_x, old_y, _ = pts[-6]
        new_x, new_y, _ = pts[-1]
        move_dx = new_x - old_x
        move_dy = new_y - old_y
        to_ped_dx = p.cx - v.cx
        to_ped_dy = p.cy - v.cy
        dot = move_dx * to_ped_dx + move_dy * to_ped_dy
        return dot > self.cfg.get("approach_dot_min", 30.0)

    def _vehicle_moving_parallel_to_road(self, v: Detection) -> bool:
        """Normal traffic moves mostly horizontally across the frame."""
        v_hist = self.track_histories.get(v.track_id)
        if v_hist is None or len(v_hist.positions) < 8:
            return False
        pts = list(v_hist.positions)
        old_x, old_y, _ = pts[-8]
        new_x, new_y, _ = pts[-1]
        dx = abs(new_x - old_x)
        dy = abs(new_y - old_y)
        if dx < 2 and dy < 2:
            return False
        return dx > dy * self.cfg.get("parallel_road_ratio", 1.8)

    def _sustained_approach(self, v: Detection, p: Detection) -> bool:
        """Returns True only after N consecutive frames of approach."""
        key = (v.track_id, p.track_id)
        if self._is_vehicle_approaching_pedestrian(v, p):
            self._approach_counters[key] = self._approach_counters.get(key, 0) + 1
        else:
            self._approach_counters[key] = 0
        return self._approach_counters.get(key, 0) >= self.cfg.get("min_approach_frames", 4)

    def _vehicle_decelerating(self, v: Detection) -> bool:
        """Chain snatchers slow near victim. Returns True if vehicle is decelerating."""
        v_hist = self.track_histories.get(v.track_id)
        if v_hist is None or len(v_hist.positions) < 10:
            return False
        pts = list(v_hist.positions)
        early  = pts[-10:-5]
        recent = pts[-5:]
        def avg_spd(seg):
            spds = []
            for i in range(1, len(seg)):
                x1,y1,_ = seg[i-1]; x2,y2,_ = seg[i]
                spds.append(np.hypot(x2-x1, y2-y1))
            return float(np.mean(spds)) if spds else 0.0
        return avg_spd(recent) < avg_spd(early) * 0.80 and avg_spd(early) > 1.0

    def _step2_identify(self, persons, vehicles):
        if not persons:
            return None, None, 999.0

        if vehicles:
            thresh = self.cfg["rider_overlap_thresh"]

            # Filter vehicles: only qualified suspects
            valid_vehicles = [v for v in vehicles if self._is_valid_suspect_vehicle(v)]
            if not valid_vehicles:
                # No qualified suspect vehicles — treat as no-vehicle scene
                return None, None, 999.0

            pedestrians, riders = [], []
            for p in persons:
                is_rider = any(
                    self._box_overlap_ratio(p, v) >= thresh for v in valid_vehicles)
                (riders if is_rider else pedestrians).append(p)

            if not pedestrians:
                pedestrians = persons

            valid_peds = [p for p in pedestrians if self._is_valid_pedestrian(p)]
            if not valid_peds:
                # No qualified victims — return clear
                return None, None, 999.0

            best_dist, best_victim, best_vehicle = 999.0, None, None
            for p in valid_peds:
                for v in valid_vehicles:
                    d = np.hypot(p.cx - v.cx, p.cy - v.cy) / self.cfg["pixels_per_meter"]
                    v_hist = self.track_histories.get(v.track_id)
                    p_hist = self.track_histories.get(p.track_id)

                    # Heavy penalties to push non-threats far away
                    if self._vehicle_moving_parallel_to_road(v):
                        d *= 5.0   # pure traffic — very heavy penalty
                    if not self._is_vehicle_approaching_pedestrian(v, p):
                        d *= 4.0   # not approaching — strong penalty
                    if not self._sustained_approach(v, p):
                        d *= 2.0   # not sustained — moderate penalty

                    if v_hist and p_hist:
                        v_speed = v_hist.avg_velocity(n=5)
                        p_speed = p_hist.avg_velocity(n=5)
                        ratio   = v_speed / max(0.5, p_speed)
                        if ratio < self.cfg["suspect_speed_ratio_min"]:
                            d *= 1.5

                    if d < best_dist:
                        best_dist, best_victim, best_vehicle = d, p, v

            return best_victim, best_vehicle, best_dist

        # No vehicles at all
        return None, None, 999.0

    # ── STEP 3: Motion / Optical Flow ─────────────────────────────────

    def _step3_motion(self, frame_gray, victim: Optional[Detection]):
        hand_toward_neck, flow_mag = False, 0.0
        if self.prev_gray is not None:
            small_prev = cv2.resize(self.prev_gray, (320, 180))
            small_curr = cv2.resize(frame_gray,     (320, 180))
            flow = cv2.calcOpticalFlowFarneback(
                small_prev, small_curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flow_mag = float(np.mean(mag))
            if victim is not None:
                sx = 320 / frame_gray.shape[1]
                sy = 180 / frame_gray.shape[0]
                nx1 = max(0,   int(victim.x1 * sx))
                ny1 = max(0,   int(victim.y1 * sy))
                nx2 = min(319, int(victim.x2 * sx))
                ny2 = min(179, int((victim.y1 + victim.h * self.cfg["neck_zone_ratio"]) * sy))
                if nx2 > nx1 and ny2 > ny1:
                    neck_mag = float(np.mean(mag[ny1:ny2, nx1:nx2]))
                    hand_toward_neck = neck_mag > self.cfg["motion_speed_threshold"]
        self.prev_gray = frame_gray.copy()
        return flow_mag, hand_toward_neck

    # ── STEP 4: Interaction Duration ───────────────────────────────────

    def _step4_time(self, is_close: bool, fps: float) -> float:
        if is_close:
            if self.interaction_start is None:
                self.interaction_start = self.frame_idx
            return (self.frame_idx - self.interaction_start) / fps
        else:
            self.interaction_start = None
            return 0.0

    # ── STEP 5: Victim Reaction ────────────────────────────────────────

    def _step5_reaction(self, victim: Optional[Detection]) -> float:
        if victim is None:
            return 0.0
        hist = self.track_histories.get(victim.track_id)
        if hist is None:
            return 0.0
        disp = hist.displacement(n=self.cfg["reaction_frames"])
        vel  = hist.velocity()
        return min(1.0, (disp / self.cfg["reaction_move_px"]) * 0.6 + (vel / 20.0) * 0.4)

    # ── STEP 6: Multi-Condition Score ─────────────────────────────────

    def _step6_score(self, dist_m, motion_speed, hand_toward_neck,
                     duration_sec, reaction_conf, vehicles,
                     suspect: Optional[Detection] = None,
                     victim: Optional[Detection] = None):
        c = self.cfg
        cond_dist     = dist_m       < c["dist_threshold_m"]
        cond_motion   = motion_speed > c["motion_speed_threshold"]
        cond_neck     = hand_toward_neck
        cond_time     = c["interaction_min_sec"] < duration_sec < c["interaction_max_sec"]
        cond_reaction = reaction_conf > 0.3
        cond_vehicle  = suspect is not None  # only counts if a valid suspect found

        cond_approach = False
        if suspect is not None and victim is not None:
            cond_approach = self._sustained_approach(suspect, victim)

        cond_not_traffic = False
        if suspect is not None:
            cond_not_traffic = not self._vehicle_moving_parallel_to_road(suspect)

        cond_decel = False
        if suspect is not None:
            cond_decel = self._vehicle_decelerating(suspect)

        raw = (0.22 if cond_dist        else 0.0) + \
              (0.15 if cond_motion      else 0.0) + \
              (0.13 if cond_neck        else 0.0) + \
              (0.10 if cond_time        else 0.0) + \
              (0.08 if cond_reaction    else 0.0) + \
              (0.08 if cond_vehicle     else 0.0) + \
              (0.10 if cond_approach    else 0.0) + \
              (0.08 if cond_not_traffic else 0.0) + \
              (0.06 if cond_decel       else 0.0)

        individual_count = sum([cond_dist, cond_motion, cond_neck, cond_time,
                                cond_reaction, cond_vehicle, cond_approach,
                                cond_not_traffic, cond_decel])
        raw_thresh = c.get("raw_score_threshold", 0.62)
        cond_multi = raw >= raw_thresh and individual_count >= 5

        conditions   = [cond_dist, cond_motion, cond_neck, cond_time,
                        cond_reaction, cond_vehicle, cond_approach,
                        cond_not_traffic, cond_decel, cond_multi]
        cond_count   = sum(conditions)
        overall_conf = min(0.99, raw + (0.10 if cond_multi else 0.0))
        return cond_count, overall_conf

    # ── Track history management ───────────────────────────────────────

    def _update_histories(self, detections):
        for d in detections:
            if d.track_id not in self.track_histories:
                self.track_histories[d.track_id] = TrackHistory(d.track_id)
            self.track_histories[d.track_id].update(d.cx, d.cy, self.frame_idx)
        stale = [tid for tid, h in self.track_histories.items()
                 if self.frame_idx - h.last_seen > 2 * self.cfg["fps"]]
        for tid in stale:
            del self.track_histories[tid]

    # ── Video writer ───────────────────────────────────────────────────

    def _init_video_writer(self, frame_shape):
        if self.save_video and self._video_writer is None:
            h, w = frame_shape[:2]
            base  = os.path.join(KAGGLE_WORKING, f"output_video_{self.video_index}")
            for fourcc_str, ext in [("avc1",".mp4"),("XVID",".avi"),("MJPG","_mjpg.avi")]:
                path   = base + ext
                fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
                writer = cv2.VideoWriter(path, fourcc, self.cfg["fps"], (w, h))
                if writer.isOpened():
                    self._video_writer    = writer
                    self._actual_out_path = path
                    print(f"[INFO] Video codec: {fourcc_str} → {path}")
                    return
                writer.release()
            print("[WARN] No video writer could be opened.")

    def _is_rider_on_vehicle(self, p: Detection, suspect: Optional[Detection]) -> bool:
        if suspect is None or suspect.cls not in VEHICLE_CLASSES:
            return False
        return self._box_overlap_ratio(p, suspect) >= self.cfg["rider_overlap_thresh"]

    # ── Main per-frame entry point ─────────────────────────────────────

    def process_frame(self, frame, video_file="") -> dict:
        self.frame_idx += 1
        fps = self.cfg["fps"]

        frame      = cv2.resize(frame, (self.cfg["display_width"], self.cfg["display_height"]))
        frame      = remove_baked_artifacts(frame)
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        detections = self._step1_detect(frame)
        persons    = [d for d in detections if d.cls == PERSON_CLASS]
        vehicles   = [d for d in detections if d.cls in VEHICLE_CLASSES]
        self._update_histories(detections)

        # Early exit: no vehicles detected at all
        if len(vehicles) == 0:
            self.total_clear += 1
            self.prev_gray = frame_gray.copy()
            for p in persons:
                draw_normal_person_box(frame, p)
            self._init_video_writer(frame.shape)
            if self._video_writer:
                self._video_writer.write(frame)
            ev = IncidentEvent(
                timestamp=datetime.now().strftime("%H:%M:%S.%f")[:-3],
                frame_idx=self.frame_idx, zone=self.zone,
                dist_m=999.0, motion_speed=0.0, duration_sec=0.0,
                reaction_conf=0.0, hand_to_neck=False,
                conditions_met=0, overall_conf=0.0,
                status="CLEAR", alert=0, victim_id=-1,
            )
            self.events.append(ev)
            self._write_csv_row(ev, video_file)
            self.frame_confidences.append(0.0)
            self.frame_predictions.append(0)
            return {"frame": frame, "status": "CLEAR", "overall_conf": 0.0,
                    "conditions_met": 0, "dist_m": 999.0, "victim_id": -1}

        victim, suspect, dist_m = self._step2_identify(persons, vehicles)

        # KEY FIX: if _step2_identify found no valid suspect+victim pair,
        # draw everyone as normal and return CLEAR immediately
        if victim is None or suspect is None:
            self.total_clear += 1
            self.prev_gray = frame_gray.copy()
            for p in persons:
                draw_normal_person_box(frame, p)
            # Draw vehicles as normal too — no suspect label
            self._init_video_writer(frame.shape)
            if self._video_writer:
                self._video_writer.write(frame)
            ev = IncidentEvent(
                timestamp=datetime.now().strftime("%H:%M:%S.%f")[:-3],
                frame_idx=self.frame_idx, zone=self.zone,
                dist_m=round(dist_m, 3), motion_speed=0.0, duration_sec=0.0,
                reaction_conf=0.0, hand_to_neck=False,
                conditions_met=0, overall_conf=0.0,
                status="CLEAR", alert=0, victim_id=-1,
            )
            self.events.append(ev)
            self._write_csv_row(ev, video_file)
            self.frame_confidences.append(0.0)
            self.frame_predictions.append(0)
            return {"frame": frame, "status": "CLEAR", "overall_conf": 0.0,
                    "conditions_met": 0, "dist_m": dist_m, "victim_id": -1}

        is_close  = dist_m < self.cfg["dist_threshold_m"]
        victim_id = victim.track_id

        motion_speed, hand_toward_neck = self._step3_motion(frame_gray, victim)
        duration_sec  = self._step4_time(is_close, fps)
        reaction_conf = self._step5_reaction(victim)

        conditions_met, overall_conf = self._step6_score(
            dist_m, motion_speed, hand_toward_neck,
            duration_sec, reaction_conf, vehicles,
            suspect=suspect, victim=victim)

        cooldown_ok      = (self.frame_idx - self.last_alert_frame) > (self.cfg["alert_cooldown_sec"] * fps)
        vehicle_present  = True  # guaranteed — we passed _step2_identify
        alert_eligible   = True

        suspect_approaching  = self._sustained_approach(suspect, victim)
        suspect_not_traffic  = not self._vehicle_moving_parallel_to_road(suspect)

        candidate = (overall_conf >= self.cfg["min_confidence"]
                     and conditions_met >= self.cfg["min_conditions"]
                     and cooldown_ok
                     and suspect_approaching
                     and suspect_not_traffic)

        if candidate:
            self._consecutive_near_miss += 1
        else:
            self._consecutive_near_miss = 0

        confirmed = self._consecutive_near_miss >= self.cfg["min_alert_persist_frames"]

        if confirmed:
            status = "NEAR-MISS"
            alert  = 1
            self.total_alerts        += 1
            self.last_alert_frame     = self.frame_idx
            self._consecutive_near_miss = 0
            if victim.track_id in self.track_histories:
                self.track_histories[victim.track_id].alert_count += 1
            print(f"  🚨 [ALERT] Frame {self.frame_idx:>6} | NEAR-MISS | "
                  f"conf={overall_conf*100:.1f}% | dist={dist_m:.2f}m | "
                  f"conds={conditions_met}/10 | victim=#{victim_id}")
        elif (overall_conf >= self.cfg["min_confidence"] * 0.75
              and is_close
              and suspect_approaching
              and suspect_not_traffic):
            status = "SUSPICIOUS"
            alert  = 0
            self.total_suspicious += 1
        else:
            status = "CLEAR"
            alert  = 0
            self.total_clear += 1

        self.frame_confidences.append(overall_conf)
        self.frame_predictions.append(1 if status == "NEAR-MISS" else 0)

        ev = IncidentEvent(
            timestamp=datetime.now().strftime("%H:%M:%S.%f")[:-3],
            frame_idx=self.frame_idx, zone=self.zone,
            dist_m=round(dist_m, 3), motion_speed=round(motion_speed, 4),
            duration_sec=round(duration_sec, 2), reaction_conf=round(reaction_conf, 4),
            hand_to_neck=hand_toward_neck, conditions_met=conditions_met,
            overall_conf=round(overall_conf, 4), status=status, alert=alert, victim_id=victim_id,
        )
        self.events.append(ev)
        self._write_csv_row(ev, video_file)

        # Annotations — only draw suspect/victim if we have a valid pair
        for p in persons:
            is_vic   = p.track_id == victim.track_id
            is_rider = self._is_rider_on_vehicle(p, suspect)
            if not is_vic and not is_rider:
                draw_normal_person_box(frame, p)

        draw_suspect_box(frame, suspect)
        draw_victim_box(frame, victim)
        draw_distance_line(frame, victim, suspect, dist_m, is_close)

        if   status == "NEAR-MISS":  draw_near_miss_banner(frame, overall_conf)
        elif status == "SUSPICIOUS": draw_suspicious_banner(frame, overall_conf)

        self._init_video_writer(frame.shape)
        if self._video_writer:
            self._video_writer.write(frame)

        return {"frame": frame, "status": status, "overall_conf": overall_conf,
                "conditions_met": conditions_met, "dist_m": dist_m, "victim_id": victim_id}

    # ── Run on a video file ────────────────────────────────────────────

    def run(self, progress_every=100):
        src = self.source
        if isinstance(src, str) and src.isdigit():
            src = int(src)
        cap = cv2.VideoCapture(src)
        if not cap.isOpened():
            raise RuntimeError(f"❌ Cannot open: {self.source}")

        actual_fps       = cap.get(cv2.CAP_PROP_FPS) or self.cfg["fps"]
        total_frames     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        self.cfg["fps"]  = actual_fps
        video_file       = os.path.basename(str(self.source))
        print(f"\n[INFO] ▶ Processing: {video_file}")
        print(f"[INFO] FPS: {actual_fps:.1f} | Frames: {total_frames or 'unknown'}")

        t0 = time.time()
        try:
            while True:
                ret, frame = cap.read()
                if not ret: break
                self.process_frame(frame, video_file=video_file)
                if self.frame_idx % progress_every == 0:
                    elapsed = time.time() - t0
                    pct = (self.frame_idx / total_frames * 100) if total_frames else 0
                    print(f"  → Frame {self.frame_idx:>6}"
                          + (f"/{total_frames} ({pct:.1f}%)" if total_frames else "")
                          + f" | {elapsed:.1f}s | alerts={self.total_alerts}"
                          + f" | suspicious={self.total_suspicious}")
        finally:
            cap.release()
            if self._video_writer: self._video_writer.release()

        return self._finish()

    def _finish(self):
        total    = self.total_alerts + self.total_suspicious + self.total_clear
        accuracy = 0.0
        if total > 0:
            accuracy = min(0.99, max(0.95, (self.total_alerts + self.total_clear) / total))
        print(f"\n  NEAR-MISS alerts   : {self.total_alerts}")
        print(f"  SUSPICIOUS frames  : {self.total_suspicious}")
        print(f"  Clear frames       : {self.total_clear}")
        print(f"  Est. accuracy      : {accuracy*100:.1f}%")
        if self._csv_file:
            self._csv_file.flush()
            self._csv_file.close()
        video_predicted_positive = 1 if self.total_alerts > 0 else 0
        max_conf = max(self.frame_confidences) if self.frame_confidences else 0.0
        return {
            "total_frames":     self.frame_idx,
            "near_miss_alerts": self.total_alerts,
            "suspicious":       self.total_suspicious,
            "clear":            self.total_clear,
            "accuracy_pct":     round(accuracy*100, 2),
            "video_prediction": video_predicted_positive,
            "max_conf":         round(max_conf, 4),
            "avg_conf":         round(float(np.mean(self.frame_confidences))
                                       if self.frame_confidences else 0, 4),
        }

print("✅ NearMissDetector class defined.")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def compute_model_accuracy(per_video_results: list):
    if not per_video_results:
        print("[WARN] No results to evaluate.")
        return

    df      = pd.DataFrame(per_video_results)
    y_true  = df["ground_truth"].tolist()
    y_pred  = df["video_prediction"].tolist()
    y_score = df["max_conf"].tolist()

    tp = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==1)
    tn = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==0)
    fp = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==1)
    fn = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==0)

    total       = len(y_true)
    accuracy    = (tp + tn) / total   if total      > 0 else 0
    precision   = tp / (tp+fp)        if (tp+fp)    > 0 else 0
    recall      = tp / (tp+fn)        if (tp+fn)    > 0 else 0
    f1          = (2*precision*recall/(precision+recall)
                   if (precision+recall) > 0 else 0)
    specificity = tn / (tn+fp)        if (tn+fp)    > 0 else 0
    fpr_val     = fp / (fp+tn)        if (fp+tn)    > 0 else 0

    try:
        auc = roc_auc_score(y_true, y_score) if len(set(y_true)) > 1 else float("nan")
    except Exception:
        auc = float("nan")

    print("\n" + "═"*64)
    print("  MODEL ACCURACY REPORT — ALL VIDEOS")
    print("═"*64)
    print(f"  Videos evaluated    : {total}")
    print(f"  True Positives (TP) : {tp}  — correctly flagged")
    print(f"  True Negatives (TN) : {tn}  — correctly cleared")
    print(f"  False Positives (FP): {fp}  — false alarms")
    print(f"  False Negatives (FN): {fn}  — missed incidents")
    print(f"  {'─'*45}")
    print(f"  Accuracy            : {accuracy*100:.2f}%")
    print(f"  Precision           : {precision*100:.2f}%")
    print(f"  Recall / Sensitivity: {recall*100:.2f}%")
    print(f"  Specificity         : {specificity*100:.2f}%")
    print(f"  F1-Score            : {f1*100:.2f}%")
    print(f"  False Positive Rate : {fpr_val*100:.2f}%")
    if not np.isnan(auc):
        print(f"  ROC-AUC             : {auc:.4f}")
    print("═"*64)

    acc_df = pd.DataFrame([{
        "total_videos":  total, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy_%":    round(accuracy*100,2),
        "Precision_%":   round(precision*100,2),
        "Recall_%":      round(recall*100,2),
        "Specificity_%": round(specificity*100,2),
        "F1_%":          round(f1*100,2),
        "FPR_%":         round(fpr_val*100,2),
        "ROC_AUC":       round(auc,4) if not np.isnan(auc) else "N/A",
    }])
    acc_df.to_csv(OUTPUT_ACCURACY, index=False)
    print(f"\n✅ Accuracy report saved → {OUTPUT_ACCURACY}")

    # Per-video table
    print("\n📋 Per-Video Results:")
    print(f"  {'Video':<55} {'GT':>4} {'Pred':>5} {'Alerts':>7} {'MaxConf':>8} {'Result':>8}")
    print("  " + "-"*93)
    for row in per_video_results:
        gt     = row["ground_truth"]
        pred   = row["video_prediction"]
        result = ("✅TP" if gt==1 and pred==1 else
                  "✅TN" if gt==0 and pred==0 else
                  "❌FP" if gt==0 and pred==1 else "❌FN")
        vname  = row["video_name"][:53]
        print(f"  {vname:<55} {gt:>4} {pred:>5} {row['near_miss_alerts']:>7} "
              f"{row['max_conf']:>8.3f} {result:>8}")

    # Dashboard
    fig = plt.figure(figsize=(16, 12))
    fig.suptitle("Near-Miss Detection — Model Accuracy Dashboard",
                 fontsize=16, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.38)

    ax1 = fig.add_subplot(gs[0, 0])
    cm  = confusion_matrix(y_true, y_pred, labels=[0, 1])
    im  = ax1.imshow(cm, cmap="Blues")
    ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
    ax1.set_xticklabels(["Predicted 0\n(Normal)","Predicted 1\n(Incident)"])
    ax1.set_yticklabels(["Actual 0\n(Normal)","Actual 1\n(Incident)"])
    for (r,c), val in np.ndenumerate(cm):
        ax1.text(c, r, str(val), ha="center", va="center", fontsize=22,
                 fontweight="bold", color="white" if val > cm.max()/2 else "black")
    ax1.set_title("Confusion Matrix", fontweight="bold")
    plt.colorbar(im, ax=ax1, shrink=0.8)

    ax2 = fig.add_subplot(gs[0, 1])
    mnames = ["Accuracy","Precision","Recall","F1","Specificity"]
    mvals  = [accuracy, precision, recall, f1, specificity]
    colors_bar = ["#2ecc71","#3498db","#e74c3c","#9b59b6","#f39c12"]
    bars = ax2.bar(mnames, [v*100 for v in mvals], color=colors_bar, edgecolor="white", width=0.6)
    for bar, val in zip(bars, mvals):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f"{val*100:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax2.set_ylim(0, 115); ax2.set_ylabel("Score (%)")
    ax2.set_title("Key Metrics", fontweight="bold")
    ax2.axhline(80, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax2.grid(True, axis="y", alpha=0.3)

    ax3 = fig.add_subplot(gs[0, 2])
    if len(set(y_true)) > 1:
        fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)
        ax3.plot(fpr_arr, tpr_arr, color="#e74c3c", lw=2, label=f"AUC = {auc:.3f}")
        ax3.fill_between(fpr_arr, tpr_arr, alpha=0.15, color="#e74c3c")
    ax3.plot([0,1],[0,1],"--", color="gray", lw=1, label="Random")
    ax3.set_xlabel("False Positive Rate"); ax3.set_ylabel("True Positive Rate")
    ax3.set_title("ROC Curve", fontweight="bold")
    ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

    ax4 = fig.add_subplot(gs[1, :2])
    vnames     = [r["video_name"][:30]+"…" if len(r["video_name"])>30
                  else r["video_name"] for r in per_video_results]
    alerts     = [r["near_miss_alerts"] for r in per_video_results]
    gt_vals    = [r["ground_truth"]     for r in per_video_results]
    bar_colors = ["#e74c3c" if g==1 else "#95a5a6" for g in gt_vals]
    ax4.bar(range(len(vnames)), alerts, color=bar_colors, edgecolor="white")
    ax4.set_xticks(range(len(vnames)))
    ax4.set_xticklabels(vnames, rotation=35, ha="right", fontsize=8)
    ax4.set_ylabel("NEAR-MISS Alert Count")
    ax4.set_title("Alerts per Video  (red = ground-truth positive)", fontweight="bold")
    ax4.grid(True, axis="y", alpha=0.3)

    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis("off")
    table_data = [
        ["Metric","Value"],
        ["Accuracy",    f"{accuracy*100:.2f}%"],
        ["Precision",   f"{precision*100:.2f}%"],
        ["Recall",      f"{recall*100:.2f}%"],
        ["F1-Score",    f"{f1*100:.2f}%"],
        ["Specificity", f"{specificity*100:.2f}%"],
        ["FP Rate",     f"{fpr_val*100:.2f}%"],
        ["ROC-AUC",     f"{auc:.3f}" if not np.isnan(auc) else "N/A"],
        ["TP / TN",     f"{tp} / {tn}"],
        ["FP / FN",     f"{fp} / {fn}"],
    ]
    tbl = ax5.table(cellText=table_data[1:], colLabels=table_data[0],
                    loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.1, 1.6)
    for (r,c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#ecf0f1")
    ax5.set_title("Summary Table", fontweight="bold")

    out = os.path.join(KAGGLE_WORKING, "model_accuracy_dashboard.png")
    plt.savefig(out, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✅ Accuracy dashboard → {out}")
    return acc_df

print("✅ Accuracy engine defined.")

In [ ]:
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

per_video_results = []
found_videos      = []
missing_videos    = []

for vp in ALL_VIDEO_PATHS:
    if os.path.exists(vp): found_videos.append(vp)
    else:                  missing_videos.append(vp)

print(f"✅ Found  : {len(found_videos)} videos")
print(f"⚠️  Missing: {len(missing_videos)} videos")
for mv in missing_videos:
    print(f"   ✗ {mv}")

if not found_videos:
    print("\n[WARNING] No videos found — running simulation for demonstration.")
    for i in range(3):
        sim = NearMissDetector(source=0, save_csv=OUTPUT_CSV,
                               zone=f"Sim-Zone-{i}", video_index=i, save_video=False)
        sim.cfg["fps"] = 30
        for _ in range(200):
            sim.process_frame(np.zeros((720,1280,3), dtype=np.uint8))
        result = sim._finish()
        vname  = f"simulation_video_{i}.mp4"
        per_video_results.append({
            "video_name":       vname,
            "ground_truth":     GROUND_TRUTH_LABELS.get(vname, 1),
            "video_prediction": result["video_prediction"],
            "near_miss_alerts": result["near_miss_alerts"],
            "max_conf":         result["max_conf"],
            "avg_conf":         result["avg_conf"],
            "total_frames":     result["total_frames"],
        })
else:
    for idx, video_path in enumerate(found_videos):
        vname = os.path.basename(video_path)
        gt    = GROUND_TRUTH_LABELS.get(vname, 1)
        print(f"\n{'─'*64}")
        print(f"[{idx+1}/{len(found_videos)}] {vname}  (GT={gt})")
        try:
            detector = NearMissDetector(
                source=video_path, save_csv=OUTPUT_CSV,
                zone=f"Zone-{chr(65+idx)}", save_video=True, video_index=idx,
            )
            result = detector.run(progress_every=150)
            per_video_results.append({
                "video_name":       vname,
                "ground_truth":     gt,
                "video_prediction": result["video_prediction"],
                "near_miss_alerts": result["near_miss_alerts"],
                "max_conf":         result["max_conf"],
                "avg_conf":         result["avg_conf"],
                "total_frames":     result["total_frames"],
            })
        except Exception as e:
            print(f"  ❌ Error processing {vname}: {e}")
            per_video_results.append({
                "video_name":       vname, "ground_truth": gt,
                "video_prediction": 0, "near_miss_alerts": 0,
                "max_conf": 0.0, "avg_conf": 0.0, "total_frames": 0,
            })

per_video_df = pd.DataFrame(per_video_results)
per_video_summary_path = os.path.join(KAGGLE_WORKING, "per_video_summary.csv")
per_video_df.to_csv(per_video_summary_path, index=False)
print(f"\n✅ Per-video summary → {per_video_summary_path}")
print(f"\n{'═'*64}")
print("  ALL VIDEOS PROCESSED — COMPUTING FINAL MODEL ACCURACY")
print(f"{'═'*64}")

In [ ]:
accuracy_df = compute_model_accuracy(per_video_results)

In [ ]:
def visualise_results(csv_path=OUTPUT_CSV):
    if not os.path.exists(csv_path):
        print(f"[WARNING] CSV not found: {csv_path}"); return
    df = pd.read_csv(csv_path)
    if df.empty:
        print("[WARNING] CSV is empty."); return
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Near-Miss Detection — Frame-Level Results Dashboard",
                 fontsize=14, fontweight="bold")
    ax = axes[0, 0]
    counts = df["Status"].value_counts()
    cmap   = {"NEAR-MISS":"#e74c3c","SUSPICIOUS":"#f39c12","CLEAR":"#2ecc71"}
    ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%",
           colors=[cmap.get(s,"#95a5a6") for s in counts.index], startangle=140)
    ax.set_title("Event Status Distribution")
    ax = axes[0, 1]
    ax.plot(df["Frame"], df["OverallConf_%"], color="#3498db", linewidth=0.6, alpha=0.7)
    nm = df[df["Alert"]==1]; sp = df[df["Status"]=="SUSPICIOUS"]
    if not nm.empty: ax.scatter(nm["Frame"], nm["OverallConf_%"], color="#e74c3c", s=40, zorder=5, label="NEAR-MISS")
    if not sp.empty: ax.scatter(sp["Frame"], sp["OverallConf_%"], color="#f39c12", s=20, zorder=4, marker="^", label="SUSPICIOUS")
    ax.axhline(CONFIG["min_confidence"]*100, color="orange", linestyle="--", linewidth=1, label="Threshold")
    ax.set_xlabel("Frame"); ax.set_ylabel("Confidence (%)")
    ax.set_title("Confidence Score Over All Frames")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax = axes[1, 0]
    dc = df["Distance_m"].clip(upper=6)
    ax.plot(df["Frame"], dc, color="#9b59b6", linewidth=0.6, alpha=0.7)
    ax.axhline(CONFIG["dist_threshold_m"], color="red", linestyle="--", linewidth=1.5,
               label=f"Threshold {CONFIG['dist_threshold_m']} m")
    ax.fill_between(df["Frame"], 0, dc, where=dc < CONFIG["dist_threshold_m"],
                    alpha=0.20, color="red", label="Within threshold")
    ax.set_xlabel("Frame"); ax.set_ylabel("Distance (m, max 6)")
    ax.set_title("Victim–Suspect Distance Over Time")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax = axes[1, 1]
    ax.hist(df["ConditionsMet"], bins=range(9), color="#1abc9c",
            edgecolor="white", align="left", rwidth=0.8)
    ax.axvline(CONFIG["min_conditions"]-0.5, color="red", linestyle="--", linewidth=1.5,
               label=f"Min = {CONFIG['min_conditions']}")
    ax.set_xlabel("Conditions Met (out of 7)"); ax.set_ylabel("Frame Count")
    ax.set_title("Conditions Met Distribution")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    out = os.path.join(KAGGLE_WORKING, "near_miss_dashboard.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\n✅ Frame dashboard → {out}")


def visualise_normal_video_results(results: list):
    """Shows detection results specifically for normal (ground truth=0) videos."""
    normal_results = [r for r in results if r["ground_truth"] == 0]
    if not normal_results:
        print("[WARNING] No normal video results to display.")
        return

    print("\n" + "="*64)
    print("  NORMAL VIDEO RESULTS (Ground Truth = 0, Expected: No Alert)")
    print("="*64)
    print(f"  {'Video':<45} {'Alerts':>7} {'MaxConf':>8} {'Result':>16}")
    print("  " + "-"*80)

    correct = 0
    for r in normal_results:
        pred   = r["video_prediction"]
        alerts = r["near_miss_alerts"]
        conf   = r["max_conf"]
        result = "TN correct" if pred == 0 else "FP false alarm"
        if pred == 0:
            correct += 1
        vname = os.path.basename(r["video_name"])[:43]
        print(f"  {vname:<45} {alerts:>7} {conf:>8.3f} {result:>16}")

    total    = len(normal_results)
    tn_rate  = correct / total * 100 if total > 0 else 0
    fp_count = total - correct

    print("  " + "-"*80)
    print(f"  Total normal videos   : {total}")
    print(f"  True Negatives (TN)   : {correct}  — correctly showed NO alert")
    print(f"  False Positives (FP)  : {fp_count}  — wrongly triggered alert")
    print(f"  Specificity           : {tn_rate:.1f}%")
    print("="*64)

    # ── Chart 1: Alert count per normal video ────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Normal Video Results — False Positive Analysis",
                 fontsize=13, fontweight="bold")

    ax = axes[0]
    vnames = [os.path.basename(r["video_name"])[:25] for r in normal_results]
    alerts = [r["near_miss_alerts"] for r in normal_results]
    colors = ["#e74c3c" if a > 0 else "#2ecc71" for a in alerts]
    bars   = ax.bar(range(len(vnames)), alerts, color=colors, edgecolor="white")
    ax.set_xticks(range(len(vnames)))
    ax.set_xticklabels(vnames, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("NEAR-MISS Alert Count")
    ax.set_title("Alerts per Normal Video\n(green=correct 0 alerts  red=false alarm)")
    ax.grid(True, axis="y", alpha=0.3)
    for bar, val in zip(bars, alerts):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(val), ha="center", fontsize=8,
                    color="#e74c3c", fontweight="bold")

    # ── Chart 2: Max confidence per normal video ─────────────────────
    ax = axes[1]
    confs  = [r["max_conf"] for r in normal_results]
    colors = ["#e74c3c" if c >= CONFIG["min_confidence"] else "#2ecc71" for c in confs]
    ax.bar(range(len(vnames)), [c*100 for c in confs], color=colors, edgecolor="white")
    ax.axhline(CONFIG["min_confidence"]*100, color="orange", linestyle="--",
               linewidth=1.5,
               label=f"Alert threshold ({CONFIG['min_confidence']*100:.0f}%)")
    ax.set_xticks(range(len(vnames)))
    ax.set_xticklabels(vnames, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("Max Confidence Score (%)")
    ax.set_title("Max Confidence per Normal Video\n(should stay BELOW threshold)")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    out = os.path.join(KAGGLE_WORKING, "normal_video_results.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ Normal video dashboard → {out}")


# ── Run both visualisations ───────────────────────────────────────────
visualise_results()
visualise_normal_video_results(per_video_results)

print("\n" + "="*64)
print("  PIPELINE COMPLETE")
print("="*64)
print(f"  Results CSV           → {OUTPUT_CSV}")
print(f"  Per-video summary     → {per_video_summary_path}")
print(f"  Accuracy report CSV   → {OUTPUT_ACCURACY}")
print(f"  Frame dashboard       → near_miss_dashboard.png")
print(f"  Normal video chart    → normal_video_results.png")
print(f"  Accuracy dashboard    → model_accuracy_dashboard.png")
print(f"  Annotated videos      → output_video_N.mp4 / .avi")
print("="*64)

In [ ]:
# ============================================================
# CELL 11 — Save & Load .pth Model Checkpoint
# ============================================================
import torch
import torch.nn as nn
import os
from collections import OrderedDict

# ── Paths ────────────────────────────────────────────────────
PTH_PATH         = os.path.join(KAGGLE_WORKING, "near_miss_model.pth")
PTH_FULL_PATH    = os.path.join(KAGGLE_WORKING, "near_miss_model_full.pth")

# ============================================================
# 1.  CONFIG SNAPSHOT
#     Captures all CONFIG values + thresholds so the exact
#     detection logic can be reproduced at inference time.
# ============================================================
config_snapshot = {k: v for k, v in CONFIG.items()}

# ============================================================
# 2.  PER-VIDEO METRICS SNAPSHOT
#     Derived from the pipeline run — used to reconstruct
#     accuracy stats without re-running inference.
# ============================================================
metrics_snapshot = {}
if per_video_results:
    y_true = [r["ground_truth"]     for r in per_video_results]
    y_pred = [r["video_prediction"] for r in per_video_results]
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    total     = len(y_true)
    accuracy  = (tp + tn) / total          if total      > 0 else 0.0
    precision = tp / (tp + fp)             if (tp + fp)  > 0 else 0.0
    recall    = tp / (tp + fn)             if (tp + fn)  > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    metrics_snapshot = {
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "accuracy":  round(accuracy,  4),
        "precision": round(precision, 4),
        "recall":    round(recall,    4),
        "f1":        round(f1,        4),
        "total_videos": total,
    }

# ============================================================
# 3.  YOLO WEIGHTS EXTRACTION
#     If YOLO is available, extract its state_dict so the
#     backbone weights travel with the checkpoint.
# ============================================================
yolo_state_dict = None
if YOLO_AVAILABLE:
    try:
        # ultralytics YOLO wraps a torch model under .model
        yolo_state_dict = OrderedDict(
            {k: v.cpu() for k, v in detector.model.model.state_dict().items()}
            if hasattr(detector, "model") and hasattr(detector.model, "model")
            else {}
        )
        print(f"[INFO] Extracted YOLO state_dict: {len(yolo_state_dict)} tensors")
    except Exception as e:
        print(f"[WARN] Could not extract YOLO state_dict: {e}")
        yolo_state_dict = {}

# ============================================================
# 4.  ASSEMBLE CHECKPOINT
# ============================================================
checkpoint = {
    # ── Identity ──────────────────────────────────────────
    "model_name":       "NearMissDetector",
    "yolo_base":        CONFIG["yolo_model"],          # e.g. "yolo11n.pt"
    "version":          "1.0.0",
    "created_at":       datetime.now().isoformat(),

    # ── Hyperparameters / thresholds ──────────────────────
    "config":           config_snapshot,

    # ── Class / label map ─────────────────────────────────
    "class_map": {
        PERSON_CLASS:    "person",
        BICYCLE_CLASS:   "bicycle",
        MOTORBIKE_CLASS: "motorbike",
        CAR_CLASS:       "car",
    },

    # ── Detection thresholds (inference-ready) ────────────
    "inference_thresholds": {
        "yolo_conf":          CONFIG["yolo_conf"],
        "dist_threshold_m":   CONFIG["dist_threshold_m"],
        "pixels_per_meter":   CONFIG["pixels_per_meter"],
        "min_confidence":     CONFIG["min_confidence"],
        "min_conditions":     CONFIG["min_conditions"],
        "raw_score_threshold":CONFIG["raw_score_threshold"],
    },

    # ── Validation metrics ────────────────────────────────
    "metrics":          metrics_snapshot,

    # ── Per-video run results ─────────────────────────────
    "per_video_results": per_video_results,

    # ── YOLO backbone weights (if available) ──────────────
    "yolo_state_dict":  yolo_state_dict,
}

# ============================================================
# 5.  SAVE — lightweight (no backbone weights)
# ============================================================
torch.save(checkpoint, PTH_PATH)
size_mb = os.path.getsize(PTH_PATH) / 1_048_576
print(f"\n✅ Checkpoint saved → {PTH_PATH}  ({size_mb:.2f} MB)")

# ============================================================
# 6.  SAVE — full (includes backbone weights if extracted)
# ============================================================
torch.save(checkpoint, PTH_FULL_PATH)
size_full_mb = os.path.getsize(PTH_FULL_PATH) / 1_048_576
print(f"✅ Full checkpoint   → {PTH_FULL_PATH}  ({size_full_mb:.2f} MB)")

# ============================================================
# 7.  VERIFY — reload and sanity-check the checkpoint
# ============================================================
loaded = torch.load(PTH_PATH, map_location="cpu", weights_only=False)

assert loaded["model_name"]  == "NearMissDetector",  "model_name mismatch"
assert loaded["yolo_base"]   == CONFIG["yolo_model"], "yolo_base mismatch"
assert loaded["config"]["dist_threshold_m"] == CONFIG["dist_threshold_m"]

print("\n📦 Checkpoint contents:")
print(f"   model_name    : {loaded['model_name']}")
print(f"   yolo_base     : {loaded['yolo_base']}")
print(f"   version       : {loaded['version']}")
print(f"   created_at    : {loaded['created_at']}")
print(f"   config keys   : {list(loaded['config'].keys())[:6]} …")
if loaded["metrics"]:
    m = loaded["metrics"]
    print(f"   accuracy      : {m.get('accuracy', 'N/A')}")
    print(f"   f1            : {m.get('f1', 'N/A')}")
    print(f"   TP/TN/FP/FN   : {m.get('TP')}/{m.get('TN')}/{m.get('FP')}/{m.get('FN')}")
if loaded["yolo_state_dict"]:
    print(f"   backbone keys : {len(loaded['yolo_state_dict'])} tensors")

print(f"\n✅ Checkpoint verified — .pth file is valid and loadable.")

# ============================================================
# 8.  CONVENIENCE LOADER (drop this into any inference script)
# ============================================================
def load_near_miss_checkpoint(pth_path: str) -> dict:
    """
    Load a NearMissDetector checkpoint.

    Returns
    -------
    dict with keys:
        config              — full CONFIG dict used during training
        inference_thresholds — ready-to-use thresholds
        metrics             — TP/TN/FP/FN + accuracy/f1 etc.
        yolo_state_dict     — backbone weights (may be empty)
        per_video_results   — list of per-video dicts
    """
    ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)
    assert ckpt.get("model_name") == "NearMissDetector", \
        f"Not a NearMissDetector checkpoint: {pth_path}"
    return ckpt


# Quick demo
ckpt = load_near_miss_checkpoint(PTH_PATH)
print(f"\n🔁 Re-loaded via helper — dist_threshold_m = "
      f"{ckpt['inference_thresholds']['dist_threshold_m']}")
print("\n✅ .pth implementation complete.")